In [1]:
%pip install polars

Note: you may need to restart the kernel to use updated packages.


In [2]:
import polars as pl
import os
from dataclasses import dataclass, field
from typing import Optional

In [3]:
os.chdir("/home/tsetsi/Python/financial-data-science-jupyter")
WORK_DIR = os.getcwd()
DATA = os.path.join(WORK_DIR, "data")

In [5]:
files = {
    "quotes_inc_eu": "DE0007500001_quotes_incremental.csv",
    "quotes_inc_us": "US2561631068_quotes_incremental.csv",
    "trades_eu": "DE0007500001_trades.csv",
    "trades_us": "US2561631068_trades.csv"
}

In [8]:
quotes_inc_eu_schema = {
    "trade_id": pl.Int128,
    "event_timestamp": pl.Datetime("ns"),
    "price": pl.Float64,
    "best_bid_price": pl.Float64,
    "best_ask_price": pl.Float64,
}

trades_eu_schema = {
    "trade_id": pl.Int128,
    "event_timestamp": pl.Datetime("ns")
}

quotes_inc_us_schema = {
    "trade_id": pl.Int128,
    "event_timestamp": pl.Datetime("ns")
}

trades_us_schema = {
    "trade_id": pl.Int128,
    "event_timestamp": pl.Datetime("ns")
}

In [6]:
@dataclass
class LimitOrderBook:
    file: str
    folder_path: str
    df: pl.LazyFrame = field(init=False)
    schema_override: Optional[dict] = None
    separator: str = ","

    def __post_init__(self):
        # pl.scan_csv doesn't load the file into memory immediately but only when called with the
        # .collect() method, which generally makes running the code significantly faster.
        # schema_overrides is optional and can be used to explicitly set a data type to a column,
        # but it will return an error if polars finds some kind of mismatch.
        # def get_data(file: str, separator: str = ",", schema_overrides: dict = None) -> pl.LazyFrame:
        self.df = pl.scan_csv(
            source=f"{self.folder_path}/{self.file}",
            separator=self.separator,
            schema_overrides=self.schema_override
        )

In [9]:
## EU: Incremental quotes
quotes_inc_eu = LimitOrderBook(
    file=files["quotes_inc_eu"],
    folder_path=DATA,
    schema_override=quotes_inc_eu_schema
)

In [15]:
MARKET_STATES = [
    "OPENING_AUCTION",
    "CONTINUOUS_TRADING", 
    "INTRADAY_AUCTION", 
    "CLOSING_AUCTION",
]

quotes_inc_eu = (
    quotes_inc_eu.df
        .filter(
            pl.col("market_state").is_in(MARKET_STATES)
        )
        .sort(by=pl.col("event_timestamp"))
)

In [19]:
# Sanity check: No original_order_id's exist within more than one venue.
print(
    quotes_inc_eu.group_by("original_order_id") \
        .agg([
            pl.col("venue").unique().alias("venues")
        ])
        .filter(pl.col("venues").list.len() > 1)
        .collect()
)

shape: (0, 2)
┌───────────────────┬───────────┐
│ original_order_id ┆ venues    │
│ ---               ┆ ---       │
│ i64               ┆ list[str] │
╞═══════════════════╪═══════════╡
└───────────────────┴───────────┘


In [17]:
quotes_aggs = [
    pl.col("venue") \
        .first()
        .alias("venue"),
    pl.col("event_timestamp") \
        .filter(pl.col("lob_action")=="INSERT")
        .first()
        .alias("insertion_date"),
    pl.col("event_timestamp") \
        .filter(pl.col("lob_action")=="REMOVE")
        .first()
        .alias("removal_date"),
    pl.col("event_timestamp") \
        .max()
        .alias("latest_event_timestamp"),
    pl.col("lob_action") \
        .eq("UPDATE")
        .sum()
        .alias("number_of_updates"),
    pl.col("price") \
        .filter(pl.col("lob_action")=="INSERT")
        .first()
        .alias("price_at_insertion"),
    pl.col("size") \
        .filter(pl.col("lob_action")=="INSERT")
        .first()
        .alias("size_at_insertion"),
    pl.col("execution_size") \
        .filter(pl.col("order_executed")==True)
        .sum() # .first()
        .alias("execution_size"),
    pl.col("price_level") \
        .filter(pl.col("lob_action")=="INSERT")
        .first()
        .alias("insertion_level"),
    pl.col("best_bid_price") \
        .filter(pl.col("lob_action")=="INSERT")
        .first()
        .alias("best_bid_price_at_insertion"),
    pl.col("best_ask_price") \
        .filter(pl.col("lob_action")=="INSERT")
        .first()
        .alias("best_ask_price_at_insertion"),
    pl.struct(
        [
            pl.col("event_timestamp") \
                .filter(pl.col("lob_action")=="UPDATE")
                .alias("lob_updates"),
            pl.col("price") \
                .filter(pl.col("lob_action")=="UPDATE")
                .alias("updated_prices"),
            pl.col("size") \
                .filter(pl.col("lob_action")=="UPDATE")
                .alias("updated_sizes"),
        ]),
]

quotes_calc = [
    (
        pl.when(pl.col("removal_date").is_not_null())
            .then(pl.col("removal_date"))
            .otherwise(pl.col("latest_event_timestamp"))
        - pl.col("insertion_date")
    ).alias("order_lifetime"),

    # (best bid price + best ask price) / 2
    (
        (
            pl.col("best_bid_price_at_insertion")
            + pl.col("best_ask_price_at_insertion")
        ) / 2
    ).alias("midpoint_at_insertion"),
]

In [44]:
quotes_collapsed = (
    quotes_inc_eu
    .group_by("original_order_id")
    .agg(quotes_aggs)
    .with_columns(quotes_calc)
)

quotes_collapsed = quotes_collapsed.with_columns([
    # Absolute distance to midpoint:
    # |Price at insertion - Midpoint at Insertion|
    (
        pl.col("price_at_insertion")
        .sub(pl.col("midpoint_at_insertion"))
        .abs()
        .alias("abs_distance_to_midpoint")
    ),
    # Relative distance to midpoint:
    # |Price at insertion - Midpoint at Insertion|
    # / Midpoint at Insertion
    (
        (
            pl.col("price_at_insertion")
            .sub(pl.col("midpoint_at_insertion"))
            .abs()
        )
        .truediv(pl.col("midpoint_at_insertion"))
        .alias("rel_distance_to_midpoint")
    ),
])

In [50]:
quotes_collapsed.collect().show(limit=10)

original_order_id,venue,insertion_date,removal_date,latest_event_timestamp,number_of_updates,price_at_insertion,size_at_insertion,execution_size,insertion_level,best_bid_price_at_insertion,best_ask_price_at_insertion,lob_updates,order_lifetime,midpoint_at_insertion,abs_distance_to_midpoint,rel_distance_to_midpoint
i64,str,datetime[ns],datetime[ns],datetime[ns],u32,f64,i64,i64,i64,f64,f64,list[struct[3]],duration[ns],f64,f64,f64
1808642,"""AQEU""",2023-09-01 07:35:28.465336,2023-09-01 07:35:28.679471,2023-09-01 07:35:28.679471,0,7.19,750,0,1,7.19,7.198,[],214135µs,7.194,0.004,0.000556
1693568048030978470,"""XETR""",2023-09-01 11:34:08.030986772,2023-09-01 11:37:47.440883052,2023-09-01 11:37:47.440883052,0,7.316,378,0,1,7.312,7.316,[],3m 39s 409896280ns,7.314,0.002,0.000273
1693551842435537687,"""XETR""",2023-09-01 07:04:02.435547693,2023-09-01 07:05:28.411762003,2023-09-01 07:05:28.411762003,0,7.112,732,0,2,7.1,7.11,[],1m 25s 976214310ns,7.105,0.007,0.000985
1693552716338630241,"""XETR""",2023-09-01 07:18:36.338639327,2023-09-01 07:18:40.691564676,2023-09-01 07:18:40.691564676,0,7.114,473,0,1,7.108,7.114,[],4s 352925349ns,7.111,0.003,0.000422
1081350376585603421,"""CEUX""",2023-09-01 08:18:21.087587,2023-09-01 08:18:23.218802,2023-09-01 08:18:23.218802,0,7.25,108,0,9,7.266,7.276,[],2s 131215µs,7.271,0.021,0.002888
2692691,"""AQEU""",2023-09-01 07:55:10.079906,2023-09-01 07:55:10.081939,2023-09-01 07:55:10.081939,0,7.27,1965,0,1,7.27,7.282,[],2033µs,7.276,0.006,0.000825
1693554534140113219,"""XETR""",2023-09-01 07:48:54.140124770,2023-09-01 11:00:00.096401017,2023-09-01 13:04:22.434073720,0,7.226,5,0,31,7.3,7.304,[],3h 11m 5s 956276247ns,7.302,0.076,0.010408
24137338,"""AQEU""",2023-09-01 15:22:57.414994,2023-09-01 15:22:57.419670,2023-09-01 15:22:57.419670,0,7.384,300,0,1,7.38,7.384,[],4676µs,7.382,0.002,0.000271
213762964011295325,"""TQEX""",2023-09-01 07:47:01.767631570,2023-09-01 07:47:01.775530010,2023-09-01 07:47:01.775530010,0,7.318,845,0,2,7.292,7.302,[],7898440ns,7.297,0.021,0.002878
